# V1-04: OpenDART 데이터 계약 확인

이 notebook은 V1에서 필요한 OpenDART 재무 데이터가 원하는 구조로 들어오는지 확인합니다. 성능 평가, LLM, LangChain, LangGraph, RAG는 사용하지 않습니다.

확인 흐름: `기업명 → 상장·업종 판정 → CFS/OFS 기준 확인 → 9개 원천 계정 매핑 → 결측 확인`

## 확인 기준

- 원하는 계정이 실제 OpenDART 행에 매핑되는가
- 금융·보험업을 V1 대상에서 제외하는 업종 판정 후보가 맞는가
- CFS와 OFS가 각각 어떤 수치를 반환하는지, 기본 정책이 CFS 하나만 선택하는지
- CFS가 없을 때만 상태 코드 `013`(데이터 없음)을 근거로 OFS로 대체하는가
- 기업·사업연도·재무제표 기준·원 계정명·원 금액이 함께 남는가
- 결측은 `None`으로 남고 임의 숫자로 바뀌지 않는가

이 notebook은 A/B 비교나 결과 보고서를 만들지 않습니다. 출력이 맞지 않으면 매핑 후보를 수정하고 다시 실행합니다.

## 1. 환경과 기존 OpenDART 호출 함수 불러오기

OpenDART 요청 로직은 notebook에 다시 만들지 않고 `src/dart/client.py`를 불러옵니다.

In [19]:
import json
import os
import sys
from pathlib import Path

In [20]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

print('OpenDART API 키를 읽었습니다. 값은 출력하지 않습니다.')

OpenDART API 키를 읽었습니다. 값은 출력하지 않습니다.


In [21]:
sys.path.insert(0, str(project_root / 'src'))

from dart.client import _get_bytes, fetch_financial_rows, find_corp_info

## 2. 확인할 기업·사업연도

아래 목록은 매핑을 확인할 표본일 뿐, V1의 기업 수 제한이 아닙니다. 필요하면 기업명과 사업연도를 추가해 다시 실행합니다.

In [22]:
COMPANY_NAMES = ['삼성전자', 'SK하이닉스', '현대자동차']
BUSINESS_YEARS = [2023, 2024]
REPORT_CODE = '11011'  # 사업보고서

print('기업:', COMPANY_NAMES)
print('사업연도:', BUSINESS_YEARS)

기업: ['삼성전자', 'SK하이닉스', '현대자동차']
사업연도: [2023, 2024]


## 3. 기업 식별과 상장·업종 정보 확인

`corp_code`는 재무 API의 필수 인자입니다. `corp_cls`, `induty_code`, `acc_mt`는 상장·비금융 판정과 결산월 확인에 사용할 원천 필드입니다.

In [23]:
COMPANY_URL = 'https://opendart.fss.or.kr/api/company.json'

def fetch_company_overview(corp_code):
    response = json.loads(
        _get_bytes(COMPANY_URL, {'crtfc_key': api_key, 'corp_code': corp_code}).decode('utf-8')
    )
    if response.get('status') != '000':
        raise RuntimeError(f"기업개황 오류 [{response.get('status')}]: {response.get('message')}")
    return response

In [6]:
company_infos = {name: find_corp_info(name, api_key) for name in COMPANY_NAMES}
company_overviews = {
    name: fetch_company_overview(info['corp_code'])
    for name, info in company_infos.items()
}

[
    {
        '기업명': name,
        'corp_code': info['corp_code'],
        'stock_code': overview.get('stock_code'),
        'corp_cls': overview.get('corp_cls'),
        'induty_code': overview.get('induty_code'),
        'acc_mt': overview.get('acc_mt'),
    }
    for name, info in company_infos.items()
    for overview in [company_overviews[name]]
]

[{'기업명': '삼성전자',
  'corp_code': '00126380',
  'stock_code': '005930',
  'corp_cls': 'Y',
  'induty_code': '264',
  'acc_mt': '12'},
 {'기업명': 'SK하이닉스',
  'corp_code': '00164779',
  'stock_code': '000660',
  'corp_cls': 'Y',
  'induty_code': '2612',
  'acc_mt': '12'},
 {'기업명': '현대자동차',
  'corp_code': '00164742',
  'stock_code': '005380',
  'corp_cls': 'Y',
  'induty_code': '30121',
  'acc_mt': '12'}]

### 3-1. 비금융 대상 판정 후보

현재 후보는 `induty_code` 앞 두 자리가 한국표준산업분류의 금융·보험업 범위인 `64`, `65`, `66`이면 V1 대상에서 제외하는 방식입니다. 삼성전자·SK하이닉스·현대자동차는 허용되고, 금융업 예시인 KB금융은 제외되어야 합니다. 실제 출력이 다르면 이 매핑을 확정하지 않습니다.

In [24]:
FINANCIAL_CHECK_COMPANY = 'KB금융'
financial_check_info = find_corp_info(FINANCIAL_CHECK_COMPANY, api_key)
financial_check_overview = fetch_company_overview(financial_check_info['corp_code'])

def classify_v1_scope(overview):
    induty_code = (overview.get('induty_code') or '').strip()
    if not induty_code.isdigit():
        return '판정불가'
    return '제외: 금융·보험업' if induty_code[:2] in {'64', '65', '66'} else '허용: 비금융업'

In [25]:
scope_overviews = {**company_overviews, FINANCIAL_CHECK_COMPANY: financial_check_overview}

[
    {
        '기업명': name,
        'induty_code': overview.get('induty_code'),
        'V1 판정': classify_v1_scope(overview),
    }
    for name, overview in scope_overviews.items()
]

[{'기업명': '삼성전자', 'induty_code': '264', 'V1 판정': '허용: 비금융업'},
 {'기업명': 'SK하이닉스', 'induty_code': '2612', 'V1 판정': '허용: 비금융업'},
 {'기업명': '현대자동차', 'induty_code': '30121', 'V1 판정': '허용: 비금융업'},
 {'기업명': 'KB금융', 'induty_code': '64992', 'V1 판정': '제외: 금융·보험업'}]

## 4. CFS 우선·OFS 대체 규칙

연결재무제표 요청이 `013`(데이터 없음)일 때만 별도재무제표를 시도합니다. 네트워크·인증·요청 오류를 OFS로 숨기지 않기 위해 다른 오류는 그대로 중단합니다.

In [26]:
def fetch_rows_prefer_cfs(corp_code, business_year):
    try:
        return 'CFS', fetch_financial_rows(corp_code, business_year, api_key, REPORT_CODE, 'CFS')
    except RuntimeError as error:
        if '[013]' not in str(error):
            raise
        return 'OFS', fetch_financial_rows(corp_code, business_year, api_key, REPORT_CODE, 'OFS')

## 5. V1 원천 계정 매핑 후보

계정 ID를 먼저 찾고, 없을 때만 계정명으로 찾습니다. 아래 목록은 확정된 구현 코드가 아니라 raw 응답에서 실제 ID·이름·재무제표 구분을 확인하기 위한 후보입니다. `None`이 나오면 다음 셀의 raw 후보 행을 보고 보완합니다.

In [27]:
TARGETS = [
    {'label': '매출액', 'sj_div': {'IS', 'CIS'}, 'ids': {'ifrs-full_Revenue', 'ifrs-full_RevenueFromContractsWithCustomers'}, 'names': {'매출액', '수익(매출액)'}},
    {'label': '영업이익', 'sj_div': {'IS', 'CIS'}, 'ids': {'dart_OperatingIncomeLoss'}, 'names': {'영업이익', '영업이익(손실)'}},
    {'label': '당기순이익', 'sj_div': {'IS', 'CIS'}, 'ids': {'ifrs-full_ProfitLoss'}, 'names': {'당기순이익', '당기순이익(손실)'}},
    {'label': '자산총계', 'sj_div': {'BS'}, 'ids': {'ifrs-full_Assets'}, 'names': {'자산총계'}},
    {'label': '부채총계', 'sj_div': {'BS'}, 'ids': {'ifrs-full_Liabilities'}, 'names': {'부채총계'}},
    {'label': '자본총계', 'sj_div': {'BS'}, 'ids': {'ifrs-full_Equity'}, 'names': {'자본총계'}},
    {'label': '유동자산', 'sj_div': {'BS'}, 'ids': {'ifrs-full_CurrentAssets'}, 'names': {'유동자산'}},
    {'label': '유동부채', 'sj_div': {'BS'}, 'ids': {'ifrs-full_CurrentLiabilities'}, 'names': {'유동부채'}},
    {'label': '영업활동현금흐름', 'sj_div': {'CF'}, 'ids': {'ifrs-full_CashFlowsFromUsedInOperatingActivities'}, 'names': {'영업활동으로인한현금흐름', '영업활동 현금흐름'}},
]

In [28]:
def find_target_row(rows, target):
    candidates = [row for row in rows if row.get('sj_div') in target['sj_div']]
    for row in candidates:
        if row.get('account_id') in target['ids']:
            return row
    return next((row for row in candidates if row.get('account_nm') in target['names']), None)

def mapped_record(corp_name, business_year, fs_div, target, row):
    return {
        '기업명': corp_name,
        '사업연도': business_year,
        '재무제표기준': fs_div,
        '항목': target['label'],
        '금액': row.get('thstrm_amount') if row else None,
        'account_id': row.get('account_id') if row else None,
        'account_nm': row.get('account_nm') if row else None,
        'sj_div': row.get('sj_div') if row else None,
        '통화': row.get('currency') if row else None,
    }

## 6. CFS와 OFS를 각각 보면 무엇이 다른가

CFS와 OFS는 서로 다른 보고 기준입니다. 아래는 삼성전자 2024년을 두 기준으로 각각 조회해 값과 기준을 비교합니다. V1 기본 결과는 둘을 동시에 합쳐 보여주지 않고 CFS가 있으면 CFS 하나만 선택합니다. 이 셀은 CFS 부재 시 OFS 대체 경로를 검증하는 셀은 아닙니다.

In [29]:
BASIS_COMPANY = '삼성전자'
BASIS_YEAR = 2024
basis_corp_code = company_infos[BASIS_COMPANY]['corp_code']
basis_rows = {
    fs_div: fetch_financial_rows(basis_corp_code, BASIS_YEAR, api_key, REPORT_CODE, fs_div)
    for fs_div in ['CFS', 'OFS']
}

[
    {
        '항목': target['label'],
        'CFS 금액': (find_target_row(basis_rows['CFS'], target) or {}).get('thstrm_amount'),
        'OFS 금액': (find_target_row(basis_rows['OFS'], target) or {}).get('thstrm_amount'),
    }
    for target in TARGETS
]

[{'항목': '매출액', 'CFS 금액': '300870903000000', 'OFS 금액': '209052241000000'},
 {'항목': '영업이익', 'CFS 금액': '32725961000000', 'OFS 금액': '12361034000000'},
 {'항목': '당기순이익', 'CFS 금액': '34451351000000', 'OFS 금액': '23582565000000'},
 {'항목': '자산총계', 'CFS 금액': '514531948000000', 'OFS 금액': '324966127000000'},
 {'항목': '부채총계', 'CFS 금액': '112339878000000', 'OFS 금액': '88569470000000'},
 {'항목': '자본총계', 'CFS 금액': '402192070000000', 'OFS 금액': '236396657000000'},
 {'항목': '유동자산', 'CFS 금액': '227062266000000', 'OFS 금액': '82320322000000'},
 {'항목': '유동부채', 'CFS 금액': '93326299000000', 'OFS 금액': '80157976000000'},
 {'항목': '영업활동현금흐름', 'CFS 금액': '72982621000000', 'OFS 금액': '52491497000000'}]

## 7. 한 기업·한 연도 raw 매핑 확인

먼저 삼성전자 2024년의 9개 항목을 확인합니다. 특히 `account_id`, `account_nm`, `sj_div`가 기대한 기준과 맞는지 직접 봅니다.

In [30]:
SAMPLE_COMPANY = '삼성전자'
SAMPLE_YEAR = 2024

sample_fs_div, sample_rows = fetch_rows_prefer_cfs(
    company_infos[SAMPLE_COMPANY]['corp_code'], SAMPLE_YEAR
)
sample_mapping = [
    mapped_record(SAMPLE_COMPANY, SAMPLE_YEAR, sample_fs_div, target, find_target_row(sample_rows, target))
    for target in TARGETS
]
sample_mapping

[{'기업명': '삼성전자',
  '사업연도': 2024,
  '재무제표기준': 'CFS',
  '항목': '매출액',
  '금액': '300870903000000',
  'account_id': 'ifrs-full_Revenue',
  'account_nm': '매출액',
  'sj_div': 'IS',
  '통화': 'KRW'},
 {'기업명': '삼성전자',
  '사업연도': 2024,
  '재무제표기준': 'CFS',
  '항목': '영업이익',
  '금액': '32725961000000',
  'account_id': 'dart_OperatingIncomeLoss',
  'account_nm': '영업이익',
  'sj_div': 'IS',
  '통화': 'KRW'},
 {'기업명': '삼성전자',
  '사업연도': 2024,
  '재무제표기준': 'CFS',
  '항목': '당기순이익',
  '금액': '34451351000000',
  'account_id': 'ifrs-full_ProfitLoss',
  'account_nm': '당기순이익',
  'sj_div': 'IS',
  '통화': 'KRW'},
 {'기업명': '삼성전자',
  '사업연도': 2024,
  '재무제표기준': 'CFS',
  '항목': '자산총계',
  '금액': '514531948000000',
  'account_id': 'ifrs-full_Assets',
  'account_nm': '자산총계',
  'sj_div': 'BS',
  '통화': 'KRW'},
 {'기업명': '삼성전자',
  '사업연도': 2024,
  '재무제표기준': 'CFS',
  '항목': '부채총계',
  '금액': '112339878000000',
  'account_id': 'ifrs-full_Liabilities',
  'account_nm': '부채총계',
  'sj_div': 'BS',
  '통화': 'KRW'},
 {'기업명': '삼성전자',
  '사업연도': 2024,
  '재무제

## 8. 다기업·다년도 구조 확인

같은 행 구조로 모든 표본을 조회합니다. 여기서 일부 값이 `None`이면 오류로 감추지 않고, 다음 셀에서 어떤 항목이 빠졌는지 확인합니다.

In [31]:
all_records = []
raw_rows_by_case = {}

for company_name, company_info in company_infos.items():
    for business_year in BUSINESS_YEARS:
        fs_div, rows = fetch_rows_prefer_cfs(company_info['corp_code'], business_year)
        raw_rows_by_case[(company_name, business_year)] = rows
        all_records.extend(
            mapped_record(company_name, business_year, fs_div, target, find_target_row(rows, target))
            for target in TARGETS
        )

all_records

[{'기업명': '삼성전자',
  '사업연도': 2023,
  '재무제표기준': 'CFS',
  '항목': '매출액',
  '금액': '258935494000000',
  'account_id': 'ifrs-full_Revenue',
  'account_nm': '영업수익',
  'sj_div': 'IS',
  '통화': 'KRW'},
 {'기업명': '삼성전자',
  '사업연도': 2023,
  '재무제표기준': 'CFS',
  '항목': '영업이익',
  '금액': '6566976000000',
  'account_id': 'dart_OperatingIncomeLoss',
  'account_nm': '영업이익',
  'sj_div': 'IS',
  '통화': 'KRW'},
 {'기업명': '삼성전자',
  '사업연도': 2023,
  '재무제표기준': 'CFS',
  '항목': '당기순이익',
  '금액': '15487100000000',
  'account_id': 'ifrs-full_ProfitLoss',
  'account_nm': '당기순이익(손실)',
  'sj_div': 'IS',
  '통화': 'KRW'},
 {'기업명': '삼성전자',
  '사업연도': 2023,
  '재무제표기준': 'CFS',
  '항목': '자산총계',
  '금액': '455905980000000',
  'account_id': 'ifrs-full_Assets',
  'account_nm': '자산총계',
  'sj_div': 'BS',
  '통화': 'KRW'},
 {'기업명': '삼성전자',
  '사업연도': 2023,
  '재무제표기준': 'CFS',
  '항목': '부채총계',
  '금액': '92228115000000',
  'account_id': 'ifrs-full_Liabilities',
  'account_nm': '부채총계',
  'sj_div': 'BS',
  '통화': 'KRW'},
 {'기업명': '삼성전자',
  '사업연도': 2023,
  '

## 9. 결측·재무제표 기준 확인

`None` 항목은 매핑 후보 또는 원본 공시를 다시 확인해야 하는 대상입니다. CFS/OFS 수는 실제로 어떤 기준이 선택됐는지 보여 줍니다.

In [32]:
missing_records = [row for row in all_records if row['금액'] is None]
basis_counts = {basis: sum(row['재무제표기준'] == basis for row in all_records) for basis in {'CFS', 'OFS'}}

print('총 매핑 행 수:', len(all_records))
print('재무제표 기준별 행 수:', basis_counts)
print('결측 행 수:', len(missing_records))
missing_records

총 매핑 행 수: 54
재무제표 기준별 행 수: {'OFS': 0, 'CFS': 54}
결측 행 수: 0


[]

## 10. 매핑되지 않은 항목의 raw 행 찾기

결측이 있으면 아래 키워드를 바꿔 원본 행의 `account_id`, `account_nm`, `sj_div`를 확인합니다. 확인된 값만 이후 `client.py`의 최종 매핑 후보에 반영합니다.

In [33]:
RAW_COMPANY = SAMPLE_COMPANY
RAW_YEAR = SAMPLE_YEAR
RAW_KEYWORD = '영업활동'

[
    {key: row.get(key) for key in ['sj_div', 'account_id', 'account_nm', 'thstrm_amount', 'currency']}
    for row in raw_rows_by_case.get((RAW_COMPANY, RAW_YEAR), sample_rows)
    if RAW_KEYWORD in (row.get('account_nm') or '')
]

[{'sj_div': 'CF',
  'account_id': 'ifrs-full_CashFlowsFromUsedInOperatingActivities',
  'account_nm': '영업활동현금흐름',
  'thstrm_amount': '72982621000000',
  'currency': 'KRW'},
 {'sj_div': 'CF',
  'account_id': 'dart_AdjustmentsForAssetsLiabilitiesOfOperatingActivities',
  'account_nm': '영업활동으로 인한 자산부채의 변동',
  'thstrm_amount': '-1567557000000',
  'currency': 'KRW'}]

In [34]:
# 11. CFS가 없을 때 OFS로 대체되는 실제 비금융 사례를 찾고, 선택 규칙을 확인합니다.
FALLBACK_CANDIDATES = ['리노공업', '국순당', '동방아그로', '신도리코', '삼화콘덴서공업']
FALLBACK_YEAR = 2024
fallback_attempts = []
fallback_result = None

for company_name in FALLBACK_CANDIDATES:
    info = find_corp_info(company_name, api_key)
    fs_div, rows = fetch_rows_prefer_cfs(info['corp_code'], FALLBACK_YEAR)
    if fs_div == 'CFS':
        fallback_attempts.append({'기업명': company_name, '선택된 기준': 'CFS', '결과': '대체 대상 아님'})
        continue

    try:
        fetch_financial_rows(info['corp_code'], FALLBACK_YEAR, api_key, REPORT_CODE, 'CFS')
    except RuntimeError as error:
        assert '[013]' in str(error), error
    else:
        raise AssertionError('OFS가 선택됐지만 CFS 부재 상태를 재현하지 못했습니다.')

    overview = fetch_company_overview(info['corp_code'])
    fallback_result = {
        '기업명': company_name,
        'induty_code': overview.get('induty_code'),
        'V1 판정': classify_v1_scope(overview),
        '선택된 기준': fs_div,
        'OFS 행 수': len(rows),
    }
    fallback_attempts.append({'기업명': company_name, '선택된 기준': 'OFS', '결과': 'CFS 없음 [013]'})
    break

print('시도 내역:', fallback_attempts)
print('확인 결과:', fallback_result)

if fallback_result is None:
    print('후보에는 CFS 부재 사례가 없었습니다. 후보를 바꿔 다시 확인해야 합니다.')
else:
    assert fallback_result['V1 판정'] == '허용: 비금융업', fallback_result
    assert fallback_result['선택된 기준'] == 'OFS', fallback_result
    assert fallback_result['OFS 행 수'] > 0, fallback_result
    print('CFS 부재 시 비금융 기업의 OFS 전체 대체 경로를 확인했습니다.')

시도 내역: [{'기업명': '리노공업', '선택된 기준': 'OFS', '결과': 'CFS 없음 [013]'}]
확인 결과: {'기업명': '리노공업', 'induty_code': '2629', 'V1 판정': '허용: 비금융업', '선택된 기준': 'OFS', 'OFS 행 수': 102}
CFS 부재 시 비금융 기업의 OFS 전체 대체 경로를 확인했습니다.


In [37]:
# 12. 금융업 세 유형(금융지주·보험·증권)이 모두 V1 대상에서 제외되는지 확인합니다.
FINANCIAL_SCOPE_CASES = {
    '금융지주': ('KB금융', '64'),
    '보험': ('삼성화재해상보험', '65'),
    '증권': ('키움증권', '66'),
}

financial_scope_results = []
for category, (company_name, expected_prefix) in FINANCIAL_SCOPE_CASES.items():
    info = find_corp_info(company_name, api_key)
    overview = fetch_company_overview(info['corp_code'])
    induty_code = (overview.get('induty_code') or '').strip()
    financial_scope_results.append({
        '유형': category,
        '기업명': company_name,
        'induty_code': induty_code,
        '기대한 앞두자리': expected_prefix,
        'V1 판정': classify_v1_scope(overview),
    })

for row in financial_scope_results:
    print(row)

assert all(row['induty_code'][:2] == row['기대한 앞두자리'] for row in financial_scope_results)
assert all(row['V1 판정'] == '제외: 금융·보험업' for row in financial_scope_results)
print('금융지주·보험·증권 사례가 모두 V1 범위에서 제외됐습니다.')

{'유형': '금융지주', '기업명': 'KB금융', 'induty_code': '64992', '기대한 앞두자리': '64', 'V1 판정': '제외: 금융·보험업'}
{'유형': '보험', '기업명': '삼성화재해상보험', 'induty_code': '65121', '기대한 앞두자리': '65', 'V1 판정': '제외: 금융·보험업'}
{'유형': '증권', '기업명': '키움증권', 'induty_code': '66121', '기대한 앞두자리': '66', 'V1 판정': '제외: 금융·보험업'}
금융지주·보험·증권 사례가 모두 V1 범위에서 제외됐습니다.


In [40]:
# 13. 서로 다른 비금융 업종에서도 9개 항목이 예상 account_id로 매핑되는지 확인합니다.
DIVERSITY_CASES = {
    'IT 서비스': 'NAVER',
    '식품': '롯데웰푸드',
    '전자부품': '리노공업',
}
DIVERSITY_YEAR = 2024
expected_ids_by_label = {target['label']: target['ids'] for target in TARGETS}
diversity_records = []

for industry, company_name in DIVERSITY_CASES.items():
    info = find_corp_info(company_name, api_key)
    overview = fetch_company_overview(info['corp_code'])
    assert classify_v1_scope(overview) == '허용: 비금융업', f'{company_name}의 V1 업종 판정 확인 필요'
    fs_div, rows = fetch_rows_prefer_cfs(info['corp_code'], DIVERSITY_YEAR)
    diversity_records.extend({
        '업종': industry,
        **mapped_record(company_name, DIVERSITY_YEAR, fs_div, target, find_target_row(rows, target)),
    } for target in TARGETS)

diversity_missing = [row for row in diversity_records if row['금액'] is None]
unexpected_account_ids = [
    row for row in diversity_records
    if row['account_id'] not in expected_ids_by_label[row['항목']]
]
print('총 매핑 행 수:', len(diversity_records))
print('결측 행 수:', len(diversity_missing))
print('예상 밖 account_id 행 수:', len(unexpected_account_ids))
for row in diversity_records:
    print({key: row[key] for key in ['업종', '기업명', '항목', '재무제표기준', 'account_id', '금액']})

assert not diversity_missing, diversity_missing
assert not unexpected_account_ids, unexpected_account_ids
print('서로 다른 세 비금융 업종에서 9개 항목이 모두 예상 account_id로 매핑됐습니다.')

총 매핑 행 수: 27
결측 행 수: 0
예상 밖 account_id 행 수: 0
{'업종': 'IT 서비스', '기업명': 'NAVER', '항목': '매출액', '재무제표기준': 'CFS', 'account_id': 'ifrs-full_Revenue', '금액': '10737719264647'}
{'업종': 'IT 서비스', '기업명': 'NAVER', '항목': '영업이익', '재무제표기준': 'CFS', 'account_id': 'dart_OperatingIncomeLoss', '금액': '1979263438827'}
{'업종': 'IT 서비스', '기업명': 'NAVER', '항목': '당기순이익', '재무제표기준': 'CFS', 'account_id': 'ifrs-full_ProfitLoss', '금액': '1931976372953'}
{'업종': 'IT 서비스', '기업명': 'NAVER', '항목': '자산총계', '재무제표기준': 'CFS', 'account_id': 'ifrs-full_Assets', '금액': '38167876036020'}
{'업종': 'IT 서비스', '기업명': 'NAVER', '항목': '부채총계', '재무제표기준': 'CFS', 'account_id': 'ifrs-full_Liabilities', '금액': '11166964187683'}
{'업종': 'IT 서비스', '기업명': 'NAVER', '항목': '자본총계', '재무제표기준': 'CFS', 'account_id': 'ifrs-full_Equity', '금액': '27000911848337'}
{'업종': 'IT 서비스', '기업명': 'NAVER', '항목': '유동자산', '재무제표기준': 'CFS', 'account_id': 'ifrs-full_CurrentAssets', '금액': '9374912034828'}
{'업종': 'IT 서비스', '기업명': 'NAVER', '항목': '유동부채', '재무제표기준': 'CFS', 'account_id': 

## 11. 실행 후 판단

- 9개 항목이 예상한 계정·재무제표 구분·금액으로 나오면: 매핑 후보를 확정 대상으로 둡니다.
- `None` 또는 다른 계정이 나오면: raw 행을 확인해 후보를 수정한 뒤 다시 실행합니다.
- 금융업 예시가 `제외: 금융·보험업`, 기존 세 기업이 `허용: 비금융업`이면: 업종 판정 후보를 다음 확인 대상으로 둡니다.
- CFS와 OFS의 금액과 기준을 함께 확인한 뒤: 기본 결과에는 CFS가 있으면 CFS 하나만 사용합니다.
- CFS가 없는 실제 사례가 나오면: OFS가 선택되고 결과에 OFS 기준이 남는지 확인합니다. 표본에 그런 사례가 없다면 OFS 대체 경로는 아직 미검증입니다.

이 확인이 끝난 뒤에만 검증된 OpenDART 수집·정규화 규칙을 `src/dart/client.py`에 반영합니다.